In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataloader import get_dataloader
import numpy as np


In [2]:
train_dataset = torch.load("trainingDataset.pt")
val_dataset   = torch.load("validationDataset.pt")
print(train_dataset)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4096, shuffle=True, pin_memory=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4096, shuffle=False, pin_memory=True, num_workers=4)
print(train_loader)

/tmp/ipykernel_56/2278383473.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load("trainingDataset.pt")


/tmp/ipykernel_56/2278383473.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  val_dataset   = torch.load("validationDataset.pt")


In [3]:
def distance_corr(
        var_1:torch.tensor,
        var_2:torch.tensor,
        normedweight:torch.tensor,
        power=1,
        )->torch.tensor:
    
    # Normalize the weights
    normedweight = normedweight/torch.sum(normedweight)*len(var_1)
    
    xx = var_1.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))
    yy = var_1.repeat(len(var_1),1).view(len(var_1),len(var_1))
    amat = (xx-yy).abs()

    xx = var_2.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))
    yy = var_2.repeat(len(var_2),1).view(len(var_2),len(var_2))
    bmat = (xx-yy).abs()

    amatavg = torch.mean(amat*normedweight,dim=1)
    Amat=amat-amatavg.repeat(len(var_1),1).view(len(var_1),len(var_1))\
        -amatavg.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))\
        +torch.mean(amatavg*normedweight)

    bmatavg = torch.mean(bmat*normedweight,dim=1)
    Bmat=bmat-bmatavg.repeat(len(var_2),1).view(len(var_2),len(var_2))\
        -bmatavg.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))\
        +torch.mean(bmatavg*normedweight)

    ABavg = torch.mean(Amat*Bmat*normedweight,dim=1)
    AAavg = torch.mean(Amat*Amat*normedweight,dim=1)
    BBavg = torch.mean(Bmat*Bmat*normedweight,dim=1)

    if(power==1):
        dCorr=(torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight)))
    elif(power==2):
        dCorr=(torch.mean(ABavg*normedweight))**2/(torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))
    else:
        dCorr=((torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))))**power
    return dCorr

class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, use_batchnorm=True, dropout=0.0):
        super().__init__()
        if not hidden_layers:
            raise ValueError("hidden_layers must contain at least one layer size")

        layers = []
        in_features = input_size
        for out_features in hidden_layers:
            layers.append(nn.Linear(in_features, out_features))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(out_features))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = out_features

        layers.append(nn.Linear(in_features, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

class ABCDModel(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_layers,
        learning_rate=1e-3,
        bce_weight=1.0,
        disco_lambda=0.0,
        flavor="single",
        use_batchnorm=True,
        dropout=0.0,
        weight_decay=1e-2,
        label_smoothing=0.0,
        use_lr_scheduler=True,
        lr_scheduler_patience=10,
        lr_scheduler_factor=0.5,
        lr_scheduler_min_lr=1e-6,
    ):
        super().__init__()

        if flavor not in {"single", "double"}:
            raise ValueError("flavor must be either 'single' or 'double'")

        self.flavor = flavor
        self.bce_weight = bce_weight
        self.disco_lambda = disco_lambda
        self.label_smoothing = label_smoothing

        self.learning_rate = learning_rate
        self.weight_decay = weight_decay

        self.use_lr_scheduler = use_lr_scheduler
        self.lr_scheduler_patience = lr_scheduler_patience
        self.lr_scheduler_factor = lr_scheduler_factor
        self.lr_scheduler_min_lr = lr_scheduler_min_lr

        # ---- model ----
        if flavor == "single":
            self.model = MLP(
                input_size=input_size,
                hidden_layers=hidden_layers,
                use_batchnorm=use_batchnorm,
                dropout=dropout,
            )
        else:
            self.model = nn.ModuleList([
                MLP(
                    input_size=input_size,
                    hidden_layers=hidden_layers,
                    use_batchnorm=use_batchnorm,
                    dropout=dropout,
                ),
                MLP(
                    input_size=input_size,
                    hidden_layers=hidden_layers,
                    use_batchnorm=use_batchnorm,
                    dropout=dropout,
                ),
            ])

    # ---------------- forward ----------------
    def forward(self, x):
        if self.flavor == "single":
            return self.model(x)

        return torch.stack(
            [self.model[0](x), self.model[1](x)],
            dim=1
        )

    # ---------------- loss ----------------
    def compute_loss(self, batch):
        data, constraint_data, labels, weights = batch

        logits = self(data)
        if logits.ndim == 1:
            logits = logits.unsqueeze(-1)

        # label smoothing
        smoothed_labels = labels.float()
        if self.label_smoothing > 0:
            smoothed_labels = (
                smoothed_labels * (1.0 - self.label_smoothing)
                + 0.5 * self.label_smoothing
            )

        scores = torch.sigmoid(logits)

        safe_weights = torch.clamp(weights, min=0.0)
        safe_weights = safe_weights / (torch.mean(safe_weights) + 1e-12)

        # BCE over heads
        bce_components = []
        for h in range(logits.shape[1]):
            bce_components.append(
                F.binary_cross_entropy_with_logits(
                    logits[:, h],
                    smoothed_labels,
                    weight=safe_weights,
                )
            )
        bce = torch.stack(bce_components).sum()

        # ---------------- disco term ----------------
        bkg_mask = labels < 0.5
        disco_term = torch.zeros((), device=logits.device)

        if bkg_mask.sum() > 0:
            bkg_scores = scores[bkg_mask]
            bkg_constraint = constraint_data[bkg_mask, 0]
            bkg_weights = safe_weights[bkg_mask]

            bkg_score_0 = bkg_scores[:, 0]

            if self.flavor == "single":
                if (
                    bkg_score_0.max() - bkg_score_0.min() > 1e-8
                    and bkg_constraint.max() - bkg_constraint.min() > 1e-8
                ):
                    disco_term = distance_corr(
                        bkg_score_0,
                        bkg_constraint,
                        bkg_weights,
                        power=2,
                    )
            else:
                bkg_score_1 = bkg_scores[:, 1]

                if (
                    bkg_score_0.max() - bkg_score_0.min() > 1e-8
                    and bkg_score_1.max() - bkg_score_1.min() > 1e-8
                ):
                    disco_term = distance_corr(
                        bkg_score_0,
                        bkg_score_1,
                        bkg_weights,
                        power=2,
                    )

        total_loss = self.bce_weight * bce + self.disco_lambda * disco_term

        return total_loss, bce, disco_term, bce_components

In [4]:
model = ABCDModel(
        input_size=16,
        hidden_layers=[64, 32, 16],
        learning_rate=0.001,
        bce_weight=1.0,
        disco_lambda=10.0,
        flavor='single',
        use_batchnorm=True,
        dropout=0.2,
        weight_decay=0.01,
        label_smoothing=0.0,
        use_lr_scheduler=True,
        lr_scheduler_patience=5,
        lr_scheduler_factor=0.5,
        lr_scheduler_min_lr=1e-6,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=model.learning_rate,
    weight_decay=model.weight_decay,
)

scheduler = None
if model.use_lr_scheduler:
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=model.lr_scheduler_factor,
        patience=model.lr_scheduler_patience,
        min_lr=model.lr_scheduler_min_lr,
    )

cuda


In [5]:
max_epochs = 100
best_val_loss = float("inf")
for epoch in range(max_epochs):

    # ---------------- TRAIN ----------------
    model.train()

    train_loss = 0.0

    for batch in train_loader:
        batch = tuple(x.to(device) for x in batch)
    
        optimizer.zero_grad()

        loss, bce, disco, _ = model.compute_loss(batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)


    # ---------------- VAL ----------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            batch = tuple(x.to(device) for x in batch)

            loss, bce, disco, _ = model.compute_loss(batch)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("Saved best model")

    # ---------------- scheduler ----------------
    if scheduler is not None:
        scheduler.step(val_loss)

Epoch 0: train=0.5660, val=0.4691
Saved best model
Epoch 1: train=0.4734, val=0.4490
Saved best model
Epoch 2: train=0.4595, val=0.4435
Saved best model
Epoch 3: train=0.4463, val=0.4454
Epoch 4: train=0.4468, val=0.4364
Saved best model
Epoch 5: train=0.4316, val=0.4353
Saved best model
Epoch 6: train=0.4377, val=0.4297
Saved best model
Epoch 7: train=0.4312, val=0.4267
Saved best model
Epoch 8: train=0.4290, val=0.4279
Epoch 9: train=0.4282, val=0.4268
Epoch 10: train=0.4258, val=0.4290
Epoch 11: train=0.4241, val=0.4236
Saved best model
Epoch 12: train=0.4191, val=0.4221
Saved best model
Epoch 13: train=0.4183, val=0.4213
Saved best model
Epoch 14: train=0.4143, val=0.4200
Saved best model
Epoch 15: train=0.4165, val=0.4166
Saved best model
Epoch 16: train=0.4089, val=0.4254
Epoch 17: train=0.4155, val=0.4160
Saved best model
Epoch 18: train=0.4116, val=0.4166
Epoch 19: train=0.4070, val=0.4152
Saved best model
Epoch 20: train=0.4047, val=0.4141
Saved best model
Epoch 21: train=0.40

In [20]:
#Evaluate the best model
def _batched_scores(model, features_tensor, batch_size, flavor, device):
    scores_0 = []
    scores_1 = []

    with torch.no_grad():
#        for start in range(0, len(features_tensor), batch_size):
        for batch, x, y, z in features_tensor:
            batch = batch.to(device)
#            batch = features_tensor[start:start + batch_size].to(device)
            logits = model(batch)
            if logits.ndim == 1:
                logits = logits.unsqueeze(-1)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            scores_0.append(probs[:, 0])
            if flavor == "double":
                scores_1.append(probs[:, 1])

    out_0 = np.concatenate(scores_0) if scores_0 else np.array([], dtype=np.float32)
    out_1 = np.concatenate(scores_1) if scores_1 else np.array([], dtype=np.float32)
    return out_0, out_1
print(type(val_dataset[0]))
print(val_dataset[0])
model.load_state_dict(torch.load("best_model.pt"))
model.to(device)
model.eval()
score_0 = (_batched_scores(model, val_loader, batch_size=4096, flavor='single', device=device))[0]
full_inference_data["dnn_score"] = score_0
_plot_roc_curves(full_inference_data, flavor='single', output_path='roc.png')

<class 'tuple'>
(tensor([0.5617, 0.3709, 0.1497, 0.0681, 0.9665, 0.9524, 0.8374, 0.8381, 0.8113,
        0.7306, 0.0602, 0.2109, 0.4639, 0.4252, 0.3956, 0.6257]), tensor([0.0258]), tensor(0.), tensor(1.5684e-07))


/tmp/ipykernel_56/3067414318.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pt"))


NameError: name 'full_inference_data' is not defined